In [8]:
%load_ext autoreload
%autoreload 2

# add tests folder to path
import sys
sys.path.append("../tests")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import importlib
import test_backward_dp_toc
importlib.reload(test_backward_dp_toc)
from test_backward_dp_toc import perform_backward_dp

V, active_eta, active_alt, active_phase_return = perform_backward_dp()

Transitions list loaded with 3259 transitions
Transitions list loaded with 3259 transitions


Topological Generations (Backward):   0%|          | 0/145 [00:00<?, ?it/s]/Volumes/CrucialX/project-equinox/src/equinox/route/batch_interpolator.py:28: UserWarning: torch.searchsorted(): boundary tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous boundary tensor if possible. This message will only appear once per program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/BucketizationUtils.h:40.)
  idx_right = torch.searchsorted(
Topological Generations (Backward):  16%|█▌        | 23/145 [00:03<00:18,  6.45it/s]


KeyboardInterrupt: 

# Inspection of Results

In [9]:
import networkx as nx
# Load the route graph
G = nx.read_gml("../data/graph/LEMD_EGLL_2023_04_01.gml")
node_to_idx = {node: i for i, node in enumerate(G.nodes())}
idx_to_node = {i: node for i, node in enumerate(G.nodes())}

In [10]:
import torch
import numpy as np

def print_states_at_node(node_id: str, active_eta: torch.Tensor, active_alt: torch.Tensor):
    tensor_eta = active_eta[node_to_idx[node_id]]
    tensor_alt = active_alt[node_to_idx[node_id]]

    # Convert tensors to numpy arrays if they're torch tensors
    arr_eta = tensor_eta.cpu().numpy() if isinstance(tensor_eta, torch.Tensor) else tensor_eta
    arr_alt = tensor_alt.cpu().numpy() if isinstance(tensor_alt, torch.Tensor) else tensor_alt

    # Print table header
    header = f"{'ETA bin':>7} | {'Climb bin Rem':>13} | {'Phase':>5} | {'ETA':>10} | {'Altitude':>10}"
    print(header)
    print('-' * len(header))

    # Iterate over all indices in the array
    for idx in np.ndindex(arr_eta.shape):
        val_eta = arr_eta[idx]
        if not np.isnan(val_eta):
            val_alt = arr_alt[idx] if not np.isnan(arr_alt[idx]) else float('nan')
            eta_bin, climb_bin, phase = idx
            print(f"{eta_bin:7d} | {climb_bin:13d} | {phase:5d} | {val_eta:10.0f} | {val_alt:10.0f}")


In [11]:
print("EGLL")
print_states_at_node("EGLL", active_eta, active_alt)
print("\n"*2)
print("LEMD")
print_states_at_node("LEMD", active_eta, active_alt)

EGLL


NameError: name 'active_eta' is not defined

In [12]:
from equinox.helpers.datetimeh import datestr_to_seconds_since_midnight
takeoff_time_ssm = datestr_to_seconds_since_midnight("2023-04-01 10:15:00")
print(f"Takeoff time: {takeoff_time_ssm}")

Takeoff time: 36900.0


# Confirming Transition List's Altitude Starting at 0

In [22]:
import pickle 
transitions_list = pickle.load(open("../data/graph/LEMD_EGLL_2023_04_01_climb_transitions.pkl", "rb")) # from test_toc_forward_dp.ipynb, to be rewritten into a more comprehensive package
print(f"Transitions list loaded with {len(transitions_list)} transitions")

Transitions list loaded with 3259 transitions


In [23]:
import numpy as np
transitions_list = np.array(transitions_list)

In [24]:
lemd_idx = node_to_idx['LEMD']
lemd_transitions = [row for row in transitions_list if row[0] == lemd_idx]
for row in lemd_transitions:
    print(row)


[1.8500e+02 0.0000e+00 0.0000e+00 5.4600e+02 1.1000e+01 1.4378e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 4.7200e+02 8.0000e+00 1.1638e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 4.9100e+02 8.0000e+00 1.0883e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 3.3100e+02 1.1000e+01 1.4575e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 3.5100e+02 1.7000e+01 2.0473e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 1.2000e+01 2.0000e+01 2.3041e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 2.5500e+02 1.9000e+01 2.2708e+04]
[1.850e+02 0.000e+00 0.000e+00 3.490e+02 2.300e+01 2.676e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 5.1900e+02 2.7000e+01 2.9448e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 2.5800e+02 1.5000e+01 1.8398e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 5.1300e+02 1.6000e+01 1.9451e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 3.3900e+02 2.0000e+01 2.3397e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 1.6000e+01 1.7000e+01 1.9963e+04]
[1.8500e+02 0.0000e+00 0.0000e+00 2.6800e+02 2.3000e+01 2.5984e+04]
[1.85e+02 0.00e+00 0.00e+00 4.47e+02 3.00e+00 3.75e+03

Looks like the altitude (third column) starts at zero. Good sign.
